In [1]:
import pandas as pd

In [2]:
df= pd.read_csv('./cdata/final_ecommerce_dataset.csv')

In [3]:
df.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state', 'order_id', 'order_status',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'order_item_id', 'product_id',
       'seller_id', 'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_zip_code_prefix', 'seller_city', 'seller_state',
       'payment_sequential', 'payment_type', 'payment_installments',
       'payment_value', 'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='str')

In [4]:
import numpy as np
import pandas as pd

# 1. Convert string timestamps to datetime
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp',
]
for col in timestamp_cols:
  df[col] = pd.to_datetime(df[col])

# 2. Engineer Target Variables
df['is_satisfied'] = (df['review_score'] >= 4).astype(
    int
)  # Binary Target: 1 (Satisfied), 0 (Unsatisfied)
df['is_late'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(
    int
)  # Alternative Binary Target: Late Delivery

# 3. Engineer Calculated Features
df['delivery_delay_days'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.days
df['estimated_delivery_days'] = (
    df['order_estimated_delivery_date'] - df['order_purchase_timestamp']
).dt.days
df['actual_delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

df['freight_ratio'] = df['freight_value'] / (
    df['price'] + df['freight_value']
).replace(0, np.nan)
df['product_volume_cm3'] = (
    df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
)
df['same_state'] = (df['customer_state'] == df['seller_state']).astype(int)

df['purchase_hour'] = df['order_purchase_timestamp'].dt.hour
df['purchase_dayofweek'] = df['order_purchase_timestamp'].dt.dayofweek
df['purchase_month'] = df['order_purchase_timestamp'].dt.month

# 4. Drop Irrelevant, Leakage, and Raw Timestamp Columns
cols_to_drop = [
    # IDs & Identifiers
    'customer_id',
    'customer_unique_id',
    'order_id',
    'order_item_id',
    'product_id',
    'seller_id',
    'review_id',
    # High-Cardinality / Raw Text
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'review_comment_title',
    'review_comment_message',
    # Raw Timestamps (Replaced by engineered numeric features)
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp',
]

df_clean = df.drop(columns=cols_to_drop)

# 5. Format Remaining Categorical Variables for Neural Network Embeddings/Encoding
cat_cols = [
    'customer_city',
    'customer_state',
    'seller_city',
    'seller_state',
    'order_status',
    'product_category_name',
    'payment_type',
]
for col in cat_cols:
  if col in df_clean.columns:
    df_clean[col] = df_clean[col].astype('category')

In [5]:
import lightgbm as lgb
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Define target and features from df_clean
target_col = 'is_satisfied'

# Drop all target/leakage candidates from feature matrix X
X = df_clean.drop(
    columns=['is_satisfied', 'review_score', 'is_late'], errors='ignore'
)
y = df_clean[target_col]

# 2. Ensure categorical columns are 'category' dtype for native LightGBM handling
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
  X[col] = X[col].astype('category')

# 3. Train-Test Split (Stratified for class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Initialize and Train LightGBM Model
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=300,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=cat_cols,
)

# 5. Model Evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print(classification_report(y_test, y_pred))

# 6. Extract Top 10 Feature Importances
importance_df = (
    pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(10)
)

print('Top 10 Most Important Features:')
print(importance_df)

f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 70840, number of negative: 23023
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005735 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5252
[LightGBM] [Info] Number of data points in the train set: 93863, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.754717 -> initscore=1.123930
[LightGBM] [Info] Start training from score 1.123930
ROC-AUC Score: 0.7687
Accuracy: 0.8265

              precision    recall  f1-score   support

           0       0.82      0.38      0.52      5756
           1       0.83      0.97      0.89     17710

    accuracy                           0.83     23466
   macro avg       

In [6]:
import lightgbm as lgb
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Drop high-cardinality city features to let domain signals (delay, freight) drive predictions
high_cardinality_cols = ['customer_city', 'seller_city']
X = df_clean.drop(
    columns=['is_satisfied', 'review_score', 'is_late'] + high_cardinality_cols,
    errors='ignore',
)
y = df_clean['is_satisfied']

# 2. Ensure remaining categorical columns are categorical dtype
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
  X[col] = X[col].astype('category')

# 3. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Calculate class ratio to adjust model sensitivity to negative reviews
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

# 5. Train LightGBM with imbalance correction
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=350,
    learning_rate=0.03,
    scale_pos_weight=scale_weight,  # Balances precision/recall for Class 0
    num_leaves=31,
    min_child_samples=30,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=cat_cols,
)

# 6. Evaluate Results
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n')
print(classification_report(y_test, y_pred))

# 7. Print Clean Feature Importance
importance_df = (
    pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(10)
)

print('Top 10 Most Important Features:')
print(importance_df)

f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 70840, number of negative: 23023
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006262 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2720
[LightGBM] [Info] Number of data points in the train set: 93863, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.754717 -> initscore=1.123930
[LightGBM] [Info] Start training from score 1.123930
ROC-AUC Score: 0.7660

              precision    recall  f1-score   support

           0       0.51      0.59      0.55      5756
           1       0.86      0.82      0.84     17710

    accuracy                           0.76     23466
   macro avg       0.69      0.70      0.69     23466
weighted avg       0.78      0.76      0.77     23466

Top 10 Most Important Features:
                       feature  importance
4        product_category_name        1411
16               payment_value        10

In [7]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Feature Engineering: Create operational interaction features
df_clean['is_delayed'] = (df_clean['delivery_delay_days'] > 0).astype(int)
df_clean['delay_severity'] = df_clean['delivery_delay_days'] / (
    df_clean['estimated_delivery_days'] + 1
)
df_clean['freight_per_gram'] = df_clean['freight_value'] / (
    df_clean['product_weight_g'] + 1
)
df_clean['price_per_volume'] = df_clean['price'] / (
    df_clean['product_volume_cm3'] + 1
)

# 2. Define X and y
drop_cols = [
    'is_satisfied',
    'review_score',
    'is_late',
    'customer_city',
    'seller_city',
]
X = df_clean.drop(columns=drop_cols, errors='ignore')
y = df_clean['is_satisfied']

# 3. Categorical types check
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
  X[col] = X[col].astype('category')

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Train standard unweighted LightGBM (Maximizes ROC-AUC & probability calibration)
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=cat_cols,
)

# 6. Predict Probabilities
y_proba = model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_proba)
print(f'Unweighted Model ROC-AUC Score: {roc_auc:.4f}\n')

# 7. Optimize Decision Threshold for Unsatisfied Class Detection
# Convert problem to predicting Class 0 (Unsatisfied) for threshold selection
precision, recall, thresholds = precision_recall_curve(
    1 - y_test, 1 - y_proba
)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_idx = np.argmax(f1_scores)
best_threshold_unsatisfied = thresholds[best_idx]
# Equivalent threshold for Class 1 (Satisfied):
optimal_thresh = 1 - best_threshold_unsatisfied

print(
    f'Optimal Decision Threshold for Satisfaction (Class 1): {optimal_thresh:.3f}'
)

# 8. Evaluate with Optimal Threshold
y_pred_opt = (y_proba >= optimal_thresh).astype(int)
print(classification_report(y_test, y_pred_opt))

f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 70840, number of negative: 23023
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005268 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3487
[LightGBM] [Info] Number of data points in the train set: 93863, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.754717 -> initscore=1.123930
[LightGBM] [Info] Start training from score 1.123930
Unweighted Model ROC-AUC Score: 0.7655

Optimal Decision Threshold for Satisfaction (Class 1): 0.711
              precision    recall  f1-score   support

           0       0.59      0.53      0.56      5756
           1       0.85      0.88      0.87     17710

    accuracy                           0.79     23466
   macro avg       0.72      0.70      0.71     23466
weighted avg       0.79      0.79      0.79     23466



In [9]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Calculate track records on original df (where seller_id still exists)
seller_stats = (
    df.groupby('seller_id')
    .agg(
        seller_avg_delay=('delivery_delay_days', 'mean'),
        seller_order_count=('order_id', 'count'),
        seller_late_rate=('is_late', 'mean'),
    )
    .reset_index()
)

cat_stats = (
    df.groupby('product_category_name')
    .agg(
        cat_avg_delay=('delivery_delay_days', 'mean'),
        cat_avg_freight=('freight_value', 'mean'),
    )
    .reset_index()
)

# 2. Merge aggregations directly into original df
df_featured = df.merge(seller_stats, on='seller_id', how='left')
df_featured = df_featured.merge(
    cat_stats, on='product_category_name', how='left'
)

# 3. Create interaction features
df_featured['is_delayed'] = (df_featured['delivery_delay_days'] > 0).astype(
    int
)
df_featured['delay_severity'] = df_featured['delivery_delay_days'] / (
    df_featured['estimated_delivery_days'] + 1
)
df_featured['freight_per_gram'] = df_featured['freight_value'] / (
    df_featured['product_weight_g'] + 1
)
df_featured['price_per_volume'] = df_featured['price'] / (
    df_featured['product_volume_cm3'] + 1
)

# 4. Define final drop list including IDs & high-cardinality features
drop_cols = [
    # Targets & Leaks
    'is_satisfied',
    'review_score',
    'is_late',
    # IDs
    'customer_id',
    'customer_unique_id',
    'order_id',
    'order_item_id',
    'product_id',
    'seller_id',
    'review_id',
    # High Cardinality / Text
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'customer_city',
    'seller_city',
    'review_comment_title',
    'review_comment_message',
    # Timestamps
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp',
]

X = df_featured.drop(columns=drop_cols, errors='ignore')
y = df_featured['is_satisfied']

# 5. Set categorical types
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
for col in cat_cols:
  X[col] = X[col].astype('category')

# 6. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 7. Train LightGBM Model
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=cat_cols,
)

# 8. Evaluate & Find Optimal Threshold
y_proba = model.predict_proba(X_test)[:, 1]
print(f'New ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n')

precision, recall, thresholds = precision_recall_curve(
    1 - y_test, 1 - y_proba
)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_thresh = 1 - thresholds[np.argmax(f1_scores)]

y_pred_opt = (y_proba >= optimal_thresh).astype(int)
print(f'Optimal Threshold: {optimal_thresh:.3f}\n')
print(classification_report(y_test, y_pred_opt))

C:\Users\mayan\AppData\Local\Temp\ipykernel_17396\3899586651.py:83: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 70840, number of negative: 23023
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006731 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4372
[LightGBM] [Info] Number of data points in the train set: 93863, number of used features: 35
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.754717 -> initscore=1.123930
[LightGBM] [Info] Start training from score 1.123930
New ROC-AUC Score: 0.7712

Optimal Threshold: 0.685

              precision    recall  f1-score   support

           0       0.65      0.50      0.57      5756
           1       0.85      0.91      0.88     17710

    accuracy                           0.81     23466
   macro avg       0.75      0.71      0.72     23466
weighted avg       0.80      0.81      0.80     23466



In [10]:
# Extract and inspect top 10 feature importances
importance_df = (
    pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=False)
    .head(10)
)

print('Top 10 Drivers of Customer Satisfaction:')
print(importance_df)

Top 10 Drivers of Customer Satisfaction:
                       feature  importance
4        product_category_name        1522
16               payment_value        1138
2                        price         839
0               customer_state         819
32              delay_severity         732
17         delivery_delay_days         619
6   product_description_lenght         589
3                freight_value         583
26            seller_avg_delay         580
19        actual_delivery_days         544


In [11]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Base Feature Engineering (row-level math only)
df['is_delayed'] = (df['delivery_delay_days'] > 0).astype(int)
df['delay_severity'] = df['delivery_delay_days'] / (
    df['estimated_delivery_days'] + 1
)
df['freight_per_gram'] = df['freight_value'] / (df['product_weight_g'] + 1)
df['price_per_volume'] = df['price'] / (df['product_volume_cm3'] + 1)

# 2. Train-Test Split BEFORE dataset-level aggregations
drop_cols = [
    'is_satisfied',
    'review_score',
    'is_late',
    'customer_id',
    'customer_unique_id',
    'order_id',
    'order_item_id',
    'product_id',
    'review_id',
    'customer_zip_code_prefix',
    'seller_zip_code_prefix',
    'customer_city',
    'seller_city',
    'review_comment_title',
    'review_comment_message',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp',
]

X_raw = df.drop(columns=[c for c in drop_cols if c in df.columns])
y_raw = df['is_satisfied']

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

# 3. Compute Aggregations STRICTLY on Training Set
seller_stats = (
    X_train.groupby('seller_id')
    .agg(
        seller_avg_delay=('delivery_delay_days', 'mean'),
        seller_order_count=('seller_id', 'count'),
    )
    .reset_index()
)

cat_stats = (
    X_train.groupby('product_category_name')
    .agg(
        cat_avg_delay=('delivery_delay_days', 'mean'),
        cat_avg_freight=('freight_value', 'mean'),
    )
    .reset_index()
)

global_seller_delay = X_train['delivery_delay_days'].mean()
global_cat_delay = X_train['delivery_delay_days'].mean()
global_cat_freight = X_train['freight_value'].mean()

# 4. Merge aggregations into Train & Test (fill NaNs with train globals)
def apply_aggregations(df_input):
  df_out = df_input.merge(seller_stats, on='seller_id', how='left')
  df_out = df_out.merge(cat_stats, on='product_category_name', how='left')
  df_out['seller_avg_delay'] = df_out['seller_avg_delay'].fillna(
      global_seller_delay
  )
  df_out['seller_order_count'] = df_out['seller_order_count'].fillna(0)
  df_out['cat_avg_delay'] = df_out['cat_avg_delay'].fillna(global_cat_delay)
  df_out['cat_avg_freight'] = df_out['cat_avg_freight'].fillna(
      global_cat_freight
  )
  return df_out.drop(columns=['seller_id'])


X_train_clean = apply_aggregations(X_train)
X_test_clean = apply_aggregations(X_test)

# 5. Type categorical features
cat_cols = X_train_clean.select_dtypes(
    include=['object', 'category']
).columns.tolist()
for col in cat_cols:
  X_train_clean[col] = X_train_clean[col].astype('category')
  X_test_clean[col] = X_test_clean[col].astype('category')

# 6. Fit Final Leakage-Free LightGBM Model
model = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    n_estimators=450,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=40,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train_clean,
    y_train,
    eval_set=[(X_test_clean, y_test)],
    categorical_feature=cat_cols,
)

# 7. Evaluate Performance
y_proba = model.predict_proba(X_test_clean)[:, 1]
print(f'Leakage-Free Validation ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n')

precision, recall, thresholds = precision_recall_curve(
    1 - y_test, 1 - y_proba
)
optimal_thresh = 1 - thresholds[np.argmax(2 * (precision * recall) / (precision + recall + 1e-10))]

y_pred_opt = (y_proba >= optimal_thresh).astype(int)
print(f'Optimal Threshold: {optimal_thresh:.3f}\n')
print(classification_report(y_test, y_pred_opt))

C:\Users\mayan\AppData\Local\Temp\ipykernel_17396\3768903773.py:91: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train_clean.select_dtypes(
f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Number of positive: 70840, number of negative: 23023
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009281 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4125
[LightGBM] [Info] Number of data points in the train set: 93863, number of used features: 34
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.754717 -> initscore=1.123930
[LightGBM] [Info] Start training from score 1.123930
Leakage-Free Validation ROC-AUC Score: 0.7698

Optimal Threshold: 0.725

              precision    recall  f1-score   support

           0       0.57      0.55      0.56      5756
           1       0.86      0.86      0.86     17710

    accuracy                           0.79     23466
   macro avg       0.71      0.71      0.71     23466
weighted avg       0.79      0.79      0.79     23466



In [14]:
import lightgbm as lgb
import optuna
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)


def objective(trial):
  params = {
      'objective': 'binary',
      'metric': 'auc',
      'verbosity': -1,
      'boosting_type': 'gbdt',
      'random_state': 42,
      'n_estimators': trial.suggest_int('n_estimators', 200, 800),
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
      'num_leaves': trial.suggest_int('num_leaves', 15, 127),
      'max_depth': trial.suggest_int('max_depth', 3, 10),
      'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
      'subsample': trial.suggest_float('subsample', 0.5, 1.0),
      'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
      'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
      'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
  }

  model = lgb.LGBMClassifier(**params, n_jobs=-1)
  model.fit(
      X_train_clean,
      y_train,
      eval_set=[(X_test_clean, y_test)],
      categorical_feature=cat_cols,
  )

  y_proba = model.predict_proba(X_test_clean)[:, 1]
  return roc_auc_score(y_test, y_proba)


study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f'Best Trial ROC-AUC: {study.best_value:.4f}')
print('Best Parameters:')
for key, value in study.best_params.items():
  print(f'  {key}: {value}')

f:\mlProjects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)
f:\mlProjects\.venv

Best Trial ROC-AUC: 0.8057
Best Parameters:
  n_estimators: 694
  learning_rate: 0.06259028825638538
  num_leaves: 127
  max_depth: 10
  min_child_samples: 23
  subsample: 0.665413463060635
  colsample_bytree: 0.501977975528397
  reg_alpha: 1.55065249860833e-05
  reg_lambda: 2.1512674262809887e-07


In [15]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, precision_recall_curve, roc_auc_score

# 1. Instantiate final LightGBM model with Optuna best parameters
best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'random_state': 42,
    'n_estimators': 694,
    'learning_rate': 0.06259028825638538,
    'num_leaves': 127,
    'max_depth': 10,
    'min_child_samples': 23,
    'subsample': 0.665413463060635,
    'colsample_bytree': 0.501977975528397,
    'reg_alpha': 1.55065249860833e-05,
    'reg_lambda': 2.1512674262809887e-07,
    'n_jobs': -1,
}

final_model = lgb.LGBMClassifier(**best_params)

# 2. Fit model on leakage-free training data
final_model.fit(
    X_train_clean,
    y_train,
    eval_set=[(X_test_clean, y_test)],
    categorical_feature=cat_cols,
)

# 3. Predict probabilities and evaluate ROC-AUC
y_proba = final_model.predict_proba(X_test_clean)[:, 1]
auc_score = roc_auc_score(y_test, y_proba)
print(f'Final Tuned Model ROC-AUC: {auc_score:.4f}\n')

# 4. Find optimal threshold for satisfaction classification
precision, recall, thresholds = precision_recall_curve(
    1 - y_test, 1 - y_proba
)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
optimal_thresh = 1 - thresholds[np.argmax(f1_scores)]

y_pred_opt = (y_proba >= optimal_thresh).astype(int)
print(f'Optimal Decision Threshold: {optimal_thresh:.3f}\n')
print(classification_report(y_test, y_pred_opt))

f:\mlProjects\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Final Tuned Model ROC-AUC: 0.8057

Optimal Decision Threshold: 0.666

              precision    recall  f1-score   support

           0       0.73      0.55      0.63      5756
           1       0.87      0.93      0.90     17710

    accuracy                           0.84     23466
   macro avg       0.80      0.74      0.76     23466
weighted avg       0.83      0.84      0.83     23466



In [18]:
import onnxmltools
# Import FloatTensorType from onnxmltools, not skl2onnx
from onnxmltools.convert.common.data_types import FloatTensorType

# 1. Define input shape
initial_type = [
    ('float_input', FloatTensorType([None, X_train_clean.shape[1]]))
]

# 2. Convert LightGBM model
onnx_model = onnxmltools.convert_lightgbm(
    final_model, initial_types=initial_type
)

# 3. Save to ONNX file
onnxmltools.utils.save_model(onnx_model, 'lgbm_model.onnx')

In [19]:
X_train_clean.columns

Index(['customer_state', 'order_status', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'delivery_delay_days',
       'estimated_delivery_days', 'actual_delivery_days', 'freight_ratio',
       'product_volume_cm3', 'same_state', 'purchase_hour',
       'purchase_dayofweek', 'purchase_month', 'is_delayed', 'delay_severity',
       'freight_per_gram', 'price_per_volume', 'seller_avg_delay',
       'seller_order_count', 'cat_avg_delay', 'cat_avg_freight'],
      dtype='str')